<a href="https://colab.research.google.com/github/Bhuvi285/Advanced-GenAI-Internship-Innomatics-Research-Labs/blob/main/Build_a_Chatbot_using_Hugging_Face_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP Assignment 3 – Chatbot using Hugging Face Transformers**

## Objective
Build a simple conversational chatbot using a pre-trained transformer model (DialoGPT) from Hugging Face that can interact with users and generate meaningful responses.

## 📚 Concept & Theory

### What are Transformers?
Transformers are a type of deep learning architecture introduced in the paper *"Attention is All You Need"* (2017). Unlike older RNN/LSTM models that process text sequentially, Transformers process all words **simultaneously** using a mechanism called **Self-Attention** — this allows them to understand context much better.

### What is DialoGPT?
- Developed by **Microsoft**
- Fine-tuned version of **GPT-2** specifically trained on **147 million Reddit conversations**
- Designed for **multi-turn dialogue** (remembers conversation history)
- Available in 3 sizes: `small`, `medium`, `large` — we use `medium` for better quality

### How the Chatbot Works
```
User Input → Tokenizer encodes text → Model generates response tokens → Decoder converts tokens back to text → Output displayed
```

### Key Parameters Used
| Parameter | Purpose |
|-----------|---------|
| `max_length` | Maximum tokens in generated response |
| `pad_token_id` | Handles padding for batch processing |
| `do_sample` | Enables probabilistic sampling (non-greedy) |
| `top_k` | Limits sampling to top-k probable tokens |
| `top_p` | Nucleus sampling — picks from top p% probability mass |
| `temperature` | Controls creativity (lower = more focused) |

In [2]:
import os
os.environ["DISABLE_TQDM"] = "1"

from transformers.utils import logging
logging.set_verbosity_error()

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging

logging.set_verbosity_error()  # 🔥 disables progress bars

In [4]:
import logging
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error() # Only show actual errors, hide warnings

In [5]:
!pip install transformers torch --quiet

In [6]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')  # Suppress unnecessary warnings

# Hugging Face Transformers imports
from transformers import AutoModelForCausalLM, AutoTokenizer

# PyTorch — used as the backend for model inference
import torch

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

✅ All libraries imported successfully!
PyTorch version: 2.10.0+cu128


## Step 2: Load Pre-trained Model and Tokenizer

We use `DialoGPT-medium` from Hugging Face Model Hub.  
- **Tokenizer**: Converts text → token IDs (numbers the model understands)  
- **Model**: The actual neural network that generates responses

In [7]:
# Define model name — using DialoGPT-medium for good quality responses
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading tokenizer for '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"⏳ Loading model for '{MODEL_NAME}'... (this may take a minute)")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode — disables dropout layers used only during training
model.eval()

print("\n✅ Model and Tokenizer loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

⏳ Loading tokenizer for 'microsoft/DialoGPT-medium'...


⏳ Loading model for 'microsoft/DialoGPT-medium'... (this may take a minute)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


✅ Model and Tokenizer loaded successfully!
Model parameters: 406,286,336


## Step 3: Define the Response Generation Function

This function:
1. Takes user input + conversation history
2. Encodes everything as tokens
3. Feeds it to the model
4. Decodes the output back to readable text

In [8]:
def generate_response(user_input, chat_history_ids):
    """
    Generates a response from DialoGPT while maintaining conversation history.
    """
    # Encode the new user input and add the End-Of-Sequence (EOS) token
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append the new user input to the chat history (if it exists)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
    else:
        bot_input_ids = new_user_input_ids

    # Create an attention mask to prevent weird, erratic responses
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate the response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,           # Prevents cutting off
        pad_token_id=tokenizer.pad_token_id,

        # --- Advanced Generation Settings to fix "weird" responses ---
        do_sample=True,            # Enable sampling so it's not robotic
        top_k=50,                  # Filter top 50 most likely words
        top_p=0.95,                # Nucleus sampling
        temperature=0.75,          # Higher = more creative, Lower = more focused
        no_repeat_ngram_size=3     # Prevents the model from repeating itself
    )

    # Decode only the newly generated tokens (ignoring the history)
    new_tokens_start = bot_input_ids.shape[-1]
    response_ids = chat_history_ids[:, new_tokens_start:]
    response = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    return response, chat_history_ids

## Step 4: Run the Chatbot

The chatbot will:
- Greet the user on startup
- Continuously accept input and generate responses
- Maintain conversation history (context-aware replies)
- Stop when user types `exit` or `quit`

> ⚠️ **Note for Colab**: The `input()` function works in Colab — just type in the text box that appears below each cell when you run it.

In [ ]:
def run_chatbot():
    """
    Main function to run the interactive chatbot loop.
    Maintains conversation history across multiple turns.
    """
    chat_history_ids = None

    print("=" * 60)
    print("        🤖 AI Chatbot — Powered by DialoGPT-medium")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("         (Type 'exit' or 'quit' to end the conversation)")
    print("-" * 60)

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            print("Chatbot: Please type something so I can help you!")
            continue

        if user_input.lower() in ['exit', 'quit']:
            print("-" * 60)
            print("Chatbot: Thank you for chatting with me! Goodbye! 👋")
            print("=" * 60)
            break

        # Generate response using the fixed function
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        print(f"Chatbot: {response}")
        print()

if __name__ == "__main__":
    run_chatbot()

        🤖 AI Chatbot — Powered by DialoGPT-medium
Chatbot: Hello! I am your AI assistant. How can I help you today?
         (Type 'exit' or 'quit' to end the conversation)
------------------------------------------------------------
You: Hello
Chatbot: Hey, how are you?

You: I am fine , how are you ?
Chatbot: Not too bad

You: But I am Bored today , aren't you ?
Chatbot: Yeah, I work at the grocery store and it's not a very interesting job

You: Oh , that's a boring job.
Chatbot: It is, but I have to do it

You: Exit
------------------------------------------------------------
Chatbot: Thank you for chatting with me! Goodbye! 👋


## 🔄 System Workflow Diagram

The diagram below illustrates the complete end-to-end flow of how the chatbot processes each user message:

- **Teal** → I/O stages (user input and output display)
- **Purple** → Tokenizer operations (encode, history, decode)
- **Coral** → Model inference (core generation step)
- **Gray** → Terminal state (chat ends)

The loop arrow on the left shows that after displaying the response, the system goes back to waiting for the next user input — until `exit` or `quit` is detected at the decision diamond.



> **Note:** The pipeline follows: User Input → Tokenize → Append History → Model.generate() → Decode → Display → [loop or exit]

chatbot_workflow_diagram.svg